# 03 · LLM client (Day 3)
Connects to Groq (`openai/gpt-oss-120b`), with request cache, retries, rate limiting,
daily-limit guard and budget cap. **Development only** — uses 20 validation comments that are
*not* in the pilot set, so the pilot stays untouched.

Done when (Dev Plan §8, Day 3): 5 live calls succeed; the second run is 100% cache hits.
Also measured here: real token counts for Sinhala / Latin / mixed comments → cost projection.

The prompt in this notebook is a **smoke-test prompt**, not the research prompt (that comes on Day 4).

In [1]:
import sys, json, datetime, asyncio
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sarcllm.llm.client import LLMClient, LLMSettings
from sarcllm.llm.ratelimit import BudgetExceeded, DailyLimitReached
from sarcllm.schemas import parse_final

settings = LLMSettings.from_env(ROOT / ".env")
for k, v in settings.public().items():
    print(f"{k:16s} {v}")

api_key          gsk_33…4A9o
base_url         https://api.groq.com/openai/v1
model            openai/gpt-oss-120b
rpm              30
tpm              8000
rpd              1000
tpd              200000
budget_usd       5.0
price_in         0.15
price_out        0.6
max_concurrency  4
timeout_s        120.0


In [2]:
client = LLMClient(settings, cache_path=ROOT / "data" / "cache" / "llm_cache.sqlite")
print("cache file:", client.store.path.relative_to(ROOT))
print("usage so far:", client.usage())

cache file: data\cache\llm_cache.sqlite
usage so far: {'all_time': {'requests': 0, 'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0}, 'today_utc': {'requests': 0, 'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0}, 'cached_responses': 0}


In [3]:
r = await client.chat([{"role": "user", "content": 'Reply with exactly this JSON object: {"ok": true}'}],
                      max_tokens=300, use_cache=False, tag="ping")
print("model returned :", r["model"])
print("content        :", r["content"])
print("finish_reason  :", r["finish_reason"])
print("reasoning      :", (r["reasoning"] or "")[:200])
print(f"tokens in/out  : {r['tokens_in']} / {r['tokens_out']}   latency {r['latency_s']}s   cost ${r['cost_usd']:.6f}")
assert r["model"].endswith("gpt-oss-120b"), "provider served a different model — check LLM_MODEL"

model returned : openai/gpt-oss-120b
content        : {"ok":true}
finish_reason  : stop
reasoning      : We just need to output that JSON.
tokens in/out  : 107 / 23   latency 36.198s   cost $0.000030


In [4]:
val = pd.read_json(ROOT / "data" / "processed" / "val.jsonl", lines=True, dtype=False)
val["id"] = val["id"].astype(str)
pilot_ids = set((ROOT / "data" / "eval_sets" / "pilot_val_300.txt").read_text(encoding="utf-8").split())
dev_path = ROOT / "data" / "eval_sets" / "dev_val_20.txt"

if dev_path.exists():
    dev_ids = dev_path.read_text(encoding="utf-8").split()
else:
    pool = val[~val.id.isin(pilot_ids)]
    # 20 rows: per script, sarcastic and non-sarcastic, spread over sentiments
    parts = []
    for (script, sarc), n in {("sinhala", 1): 4, ("sinhala", 0): 3, ("latin", 1): 4, ("latin", 0): 3,
                              ("mixed", 1): 3, ("mixed", 0): 3}.items():
        cell = pool[(pool.script_type == script) & (pool.is_sarcastic == sarc)]
        parts.append(cell.sample(frac=1, random_state=42).groupby("sentiment").head(2).head(n))
    dev_ids = sorted(pd.concat(parts).id)
    dev_path.write_text("\n".join(dev_ids) + "\n", encoding="utf-8")

dev = val.set_index("id").loc[dev_ids].reset_index()
assert not set(dev.id) & pilot_ids, "dev set overlaps the pilot"
display(pd.crosstab(dev.script_type, [dev.is_sarcastic, dev.sentiment]))

is_sarcastic        0                         1                 
sentiment    negative neutral positive negative neutral positive
script_type                                                     
latin               0       1        2        2       1        1
mixed               2       0        1        2       1        0
sinhala             2       0        1        2       2        0

In [5]:
SMOKE_PROMPT = """You analyse Sinhala / Singlish social-media comments.
Read the parent post caption (context) and the comment. Return ONLY a JSON object with keys:
literal_sentiment (negative|neutral|positive), is_sarcastic (true|false),
intended_sentiment (negative|neutral|positive), rationale (max 40 words), evidence_ids (empty list).

CONTEXT: {context}
COMMENT: {comment}"""

def smoke_messages(row):
    return [{"role": "user", "content": SMOKE_PROMPT.format(context=row.context_text, comment=row.text)}]

GEN = dict(temperature=0.0, max_tokens=1024, reasoning_effort="low", json_mode=True)

five = dev.head(5)
run1 = await client.map([smoke_messages(r) for r in five.itertuples()], tag="smoke", **GEN)

def summarise(rows, results):
    out = []
    for row, res in zip(rows.itertuples(), results):
        if isinstance(res, Exception):
            out.append({"id": row.id, "error": f"{type(res).__name__}: {res}"[:120]}); continue
        p = parse_final(res["content"])
        out.append({"id": row.id, "script": row.script_type, "gold_sarc": row.is_sarcastic,
                    "gold_sent": row.sentiment, "parse_ok": p.ok,
                    "pred_sarc": p.obj.is_sarcastic if p.ok else None,
                    "pred_sent": p.obj.intended_sentiment if p.ok else None,
                    "tok_in": res["tokens_in"], "tok_out": res["tokens_out"],
                    "latency_s": res["latency_s"], "cache_hit": res["cache_hit"], "finish": res["finish_reason"],
                    "error": "" if p.ok else p.error})
    return pd.DataFrame(out)

s1 = summarise(five, run1)
display(s1)
print(f"parse rate {s1.parse_ok.mean():.0%} · mean latency {s1.latency_s.mean():.1f}s")

,id,script,gold_sarc,gold_sent,parse_ok,pred_sarc,pred_sent,tok_in,tok_out,latency_s,cache_hit,finish,error
0,P125C80,sinhala,0,positive,True,False,positive,221,102,0.970,False,stop,
1,P137C43,mixed,1,negative,True,False,neutral,223,71,0.511,False,stop,
2,P166C94,latin,0,positive,True,False,positive,3027,123,1.113,False,stop,
3,P175C53,sinhala,0,negative,True,False,negative,220,68,0.663,False,stop,
4,P175C71,latin,1,negative,True,False,negative,200,82,0.643,False,stop,


parse rate 100% · mean latency 0.8s


In [6]:
first = next(r for r in run1 if isinstance(r, dict))
print("REASONING:\n", (first["reasoning"] or "")[:800])
print("\nCONTENT:\n", first["content"])

REASONING:
 We need to output JSON. Sentiment? Comment "සිතුවිල්ල දහස් වටි" seems incomplete maybe "picture worth thousands"? Likely positive? It's praising? Probably positive. No sarcasm. Intended same. Provide rationale.

CONTENT:
 {"literal_sentiment":"positive","is_sarcastic":false,"intended_sentiment":"positive","rationale":"The comment praises the picture as valuable, indicating a positive reaction.","evidence_ids":[]}


In [7]:
run2 = await client.map([smoke_messages(r) for r in five.itertuples()], tag="smoke", **GEN)
s2 = summarise(five, run2)
print("cache hits:", f"{s2.cache_hit.mean():.0%}")
assert s2.cache_hit.all(), "second run was not fully served from cache"
assert (s1.drop(columns=["latency_s", "cache_hit"]).fillna("").astype(str).values ==
        s2.drop(columns=["latency_s", "cache_hit"]).fillna("").astype(str).values).all()
print("Day 3 done-when: PASS (5 live calls OK, 100% cache hits on re-run)")

cache hits: 100%
Day 3 done-when: PASS (5 live calls OK, 100% cache hits on re-run)


In [8]:
WRAP = "Do nothing. Reply OK.\nCONTEXT: {context}\nCOMMENT: {comment}"
MEAS = dict(temperature=0.0, max_tokens=64, reasoning_effort="low", json_mode=False)

empty = await client.chat([{"role": "user", "content": WRAP.format(context="", comment="")}], tag="tokens", **MEAS)
base_tokens = empty["tokens_in"]

pool = val[~val.id.isin(pilot_ids) & ~val.id.isin(dev_ids)]
meas = pd.concat([g.sample(10, random_state=7) for _, g in pool.groupby("script_type")])
res = await client.map([[{"role": "user", "content": WRAP.format(context=r.context_text, comment=r.text)}]
                        for r in meas.itertuples()], tag="tokens", **MEAS)
ok = [isinstance(x, dict) for x in res]
meas = meas[ok].assign(
    tokens=[x["tokens_in"] - base_tokens for x in res if isinstance(x, dict)],
    chars=lambda d: d.text.str.len() + d.context_text.str.len(),
)
meas["chars_per_token"] = meas.chars / meas.tokens
tok = meas.groupby("script_type")[["tokens", "chars", "chars_per_token"]].mean().round(1)
display(tok)
T_ITEM = float(meas.tokens.mean())
print(f"wrapper overhead {base_tokens} tokens · mean comment+caption = {T_ITEM:.0f} tokens")

,tokens,chars,chars_per_token
script_type,,,
latin,219.8,412.9,2.0
mixed,736.9,1393.2,2.0
sinhala,282.2,512.4,1.8


wrapper overhead 84 tokens · mean comment+caption = 402 tokens


In [9]:
# --- assumptions (edit after Day 7) ---
SYS, EX_K, FEW, INSTR = 900, 8, 6, 120     # system prompt, retrieved k, few-shot n, task instructions
OUT_SINGLE, OUT_AGENT, OUT_SYNTH = 350, 300, 400   # incl. low-effort reasoning tokens
EVAL_N, HY_UNION, SARC_N, PILOT_N = 4554, 2999, 2277, 300   # option B; HY1 union from notebook 02

ex = T_ITEM + 15
single = lambda n_ex: (SYS + n_ex * ex + T_ITEM + INSTR, OUT_SINGLE)
l4_in = 3 * (SYS + EX_K * ex + T_ITEM + 2 * INSTR) + (SYS + T_ITEM + 3 * OUT_AGENT + 2 * INSTR)
l4_out = 3 * OUT_AGENT + OUT_SYNTH
per_item = {"L1": single(0), "L2": single(FEW), "L3": single(EX_K), "L4": (l4_in, l4_out), "L6": (l4_in + 30, l4_out)}

plan = [("Pilot (L1–L4, L6, HY1, HY3, L4 repeat, effort low vs medium)", None, None),
        ("L1", "L1", EVAL_N), ("L2", "L2", EVAL_N), ("L3", "L3", EVAL_N), ("L4", "L4", EVAL_N),
        ("L6", "L6", EVAL_N), ("HY1 (union of 3 seeds)", "L4", HY_UNION), ("HY3", "L3", HY_UNION),
        ("A1 (gold-sarcastic, no caption)", "L4", SARC_N)]
rows = []
for name, cond, n in plan:
    if cond is None:   # pilot: 300 × (L1+L2+L3+2·L4+L6) + routed share × (HY1+HY3) + L4 repeat
        tin = PILOT_N * (sum(per_item[c][0] for c in ("L1", "L2", "L3", "L6")) + 3 * per_item["L4"][0]
                         + 0.36 * (per_item["L4"][0] + per_item["L3"][0]))
        tout = PILOT_N * (sum(per_item[c][1] for c in ("L1", "L2", "L3", "L6")) + 3 * per_item["L4"][1]
                          + 0.36 * (per_item["L4"][1] + per_item["L3"][1]))
    else:
        tin, tout = n * per_item[cond][0], n * per_item[cond][1]
    rows.append({"stage": name, "M tokens in": tin / 1e6, "M tokens out": tout / 1e6,
                 "USD": client.cost(tin, tout)})
proj = pd.DataFrame(rows)
proj.loc[len(proj)] = ["TOTAL", proj["M tokens in"].sum(), proj["M tokens out"].sum(), proj.USD.sum()]
proj["days at free TPD"] = (proj["M tokens in"] + proj["M tokens out"]) * 1e6 / settings.tpd
display(proj.round(2))

,stage,M tokens in,M tokens out,USD,days at free TPD
0,"Pilot (L1–L4, L6, HY1, HY3, L4 repeat, effort ...",25.88,2.05,5.11,139.67
1,L1,6.47,1.59,1.93,40.34
2,L2,17.86,1.59,3.64,97.29
3,L3,21.66,1.59,4.21,116.27
4,L4,77.74,5.92,15.21,418.29
5,L6,77.87,5.92,15.23,418.97
6,HY1 (union of 3 seeds),51.19,3.90,10.02,275.46
7,HY3,14.26,1.05,2.77,76.57
8,"A1 (gold-sarcastic, no caption)",38.87,2.96,7.61,209.14
9,TOTAL,331.82,26.58,65.72,1792.00


In [10]:
row = dev.iloc[0]
reps = [await client.chat(smoke_messages(row), use_cache=False, tag="determinism", **GEN) for _ in range(3)]
parsed = [parse_final(x["content"]) for x in reps]
labels = [(p.obj.is_sarcastic, p.obj.intended_sentiment) if p.ok else None for p in parsed]
print("labels      :", labels)
print("same labels :", len(set(labels)) == 1)
print("same text   :", len({x["content"] for x in reps}) == 1)
# Different text with the same labels is normal for sampled reasoning models; the pilot measures
# label agreement on 300 comments (Dev Plan §7).

labels      : [(False, 'positive'), (False, 'positive'), (False, 'positive')]
same labels : True
same text   : True


In [11]:
u = client.usage()
display(pd.DataFrame(client.store.by_tag(), columns=["tag", "requests", "tokens_in", "tokens_out", "usd"]))
print(f"""## {datetime.date.today()} · LLM client (notebook 03)
- Provider Groq, model requested `{settings.model}`, model returned `{r['model']}`; free tier (RPM {settings.rpm}, TPM {settings.tpm}, RPD {settings.rpd}, TPD {settings.tpd}).
- Generation defaults: temperature {GEN['temperature']}, reasoning_effort {GEN['reasoning_effort']}, JSON mode, max_tokens {GEN['max_tokens']}.
- Day 3 done-when: PASS (5 live calls, parse rate {s1.parse_ok.mean():.0%}; re-run 100% cache hits).
- Dev set: data/eval_sets/dev_val_20.txt (20 val comments, disjoint from pilot).
- Measured comment+caption tokens: {tok['tokens'].to_dict()} (mean {T_ITEM:.0f}).
- Projected full-study cost ≈ ${proj.USD.iloc[-1]:.0f} ({proj['M tokens in'].iloc[-1] + proj['M tokens out'].iloc[-1]:.0f}M tokens) → Developer plan needed before the pilot.
- Usage so far: {u['all_time']['requests']} paid requests, ${u['all_time']['cost_usd']:.4f}.
""")

,tag,requests,tokens_in,tokens_out,usd
0,determinism,3,663,306,0.000283
1,ping,1,107,23,0.000030
2,smoke,5,3891,446,0.000851
3,tokens,30,14172,846,0.002633


## 2026-11-21 · LLM client (notebook 03)
- Provider Groq, model requested `openai/gpt-oss-120b`, model returned `openai/gpt-oss-120b`; free tier (RPM 30, TPM 8000, RPD 1000, TPD 200000).
- Generation defaults: temperature 0.0, reasoning_effort low, JSON mode, max_tokens 1024.
- Day 3 done-when: PASS (5 live calls, parse rate 100%; re-run 100% cache hits).
- Dev set: data/eval_sets/dev_val_20.txt (20 val comments, disjoint from pilot).
- Measured comment+caption tokens: {'latin': 219.8, 'mixed': 736.9, 'sinhala': 282.2} (mean 402).
- Projected full-study cost ≈ $66 (358M tokens) → Developer plan needed before the pilot.
- Usage so far: 39 paid requests, $0.0038.



In [12]:
tr = pd.read_json(ROOT / "data" / "processed" / "train.jsonl", lines=True, dtype=False)
lens = pd.DataFrame({"comment_chars": tr.text.str.len(), "caption_chars": tr.context_text.str.len()})
display(lens.describe(percentiles=[.5, .9, .99]).round(0))
display(tok)   # Cell 9's per-script token table

,comment_chars,caption_chars
count,32582.0,32582.0
mean,77.0,430.0
std,133.0,711.0
min,1.0,4.0
50%,42.0,112.0
90%,160.0,1420.0
99%,589.0,3053.0
max,3023.0,5693.0


,tokens,chars,chars_per_token
script_type,,,
latin,219.8,412.9,2.0
mixed,736.9,1393.2,2.0
sinhala,282.2,512.4,1.8


## 2026-11-21 · LLM client verified on Groq (notebook 03)
- openai/gpt-oss-120b returned as requested; JSON mode works; reasoning field returned; parse rate 5/5.
- Cache: re-run 100% hits. Dev set dev_val_20.txt (disjoint from pilot).
- Measured: wrapper 84 tokens; comment+caption mean 402 tokens.
- Projection (assumed templates, option B): 332M tokens ≈ $66 at list price; free tier infeasible (≈1,800 days).
- Plan: Developer plan before the pilot; Batch API (−50%) for all Gate-G2 test runs → ≈ $40.
- Pending decision (before G1): truncate captions of retrieved examples (not of the target).